### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)


2. A function or coroutine to execute.

In [3]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") #type:ignore

model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("why do parrots talk?")
response.content



'\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "why do parrots talk?" This is a common question about avian behavior, specifically parrot vocalization/mimicry.\n\n2.  **Identify Key Concepts**:\n   - Parrot vocalization/mimicry\n   - Biological/evolutionary reasons\n   - Social/behavioral drivers\n   - Cognitive mechanisms\n   - Comparison with other animals\n   - Scientific consensus vs. misconceptions\n\n3.  **Core Reasons (Scientific/Behavioral)**:\n   - **Social bonding**: Parrots are highly social flock animals. Vocal mimicry helps them integrate into groups, strengthen bonds, and communicate with mates/flock members.\n   - **Communication & Survival**: In the wild, they use vocalizations to coordinate, warn of danger, find mates, and maintain flock cohesion. Mimicry may extend this to human environments.\n   - **Cognitive ability**: Parrots have advanced neural structures (specifically, the "song system" homologous to human language are

In [4]:
from langchain.tools  import tool

@tool
def get_weather(location:str) -> str: 
    """ Get the wether at a location """
    
    return f"it's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's weather like in Boston")
print(response)

for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool : {tool_call["name"]}")
    print(f"args : {tool_call["args"]}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user is asking for the current weather in Boston.\n2.  **Identify Available Tools**: I have a `get_weather` function that takes a `location` parameter.\n3.  **Extract Parameters**: Location = "Boston".\n4.  **Call Tool**: `get_weather(location="Boston")`.\n5.  **Formulate Response**: Wait for the tool output, then construct a natural language response based on the result. Since I\'m simulating the tool call, I will just make the call.\n\nLet\'s call the function. \nNote: I should ensure the parameter matches exactly what\'s expected. `location` is a string. "Boston" is correct.\nProceed. \nOutput matches the function schema.✅\nI will generate the tool call. \n`print(get_weather(location="Boston"))` -> wait, I just output the JSON/tool call format.\nActually, the prompt says I have access to the function. I will call it.\n`get_weather(location="Boston")`\nDone. \nProceeding. \n(Self-

### Tool Execution Loops

In [6]:
#  Step 1: Model generates tool calls
message  =[{"role":"user","content":"What's the weather in Boston ?"}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)


## Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

# Step 3: Pass results back to model for final response

final_response = model_with_tools.invoke(message)
print(final_response.text)

# "The current weather in Boston is 72°F and sunny."


It's sunny in Boston.


In [7]:
message

[{'role': 'user', 'content': "What's the weather in Boston ?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify user intent: User wants to know the weather in Boston.\n2.  Identify available tools: `get_weather` function takes a `location` parameter.\n3.  Extract parameters: `location` = "Boston".\n4.  Call the function: `get_weather(location="Boston")`.\n5.  Wait for response and formulate answer based on the result. (Self-correction/Refinement: I will just call the tool now.) \nProceed. \nNo extra steps needed. Output matches tool call format.✅\n', 'tool_calls': [{'id': 'a6a0z6kdg', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 146, 'prompt_tokens': 276, 'total_tokens': 422, 'completion_time': 0.329663465, 'completion_tokens_details': {'reasoning_tokens': 118}, 'prompt_time': 0.023205373, 'prompt_tokens_details': None, 'queue_time